# Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Load and preprocess the CIFAR-10 dataset


In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 [==============================] - 2s 0us/step


# Resize the images to match MobileNet input dimensions (224x224)


In [3]:
x_train_resized = tf.image.resize(x_train, (224, 224))
x_test_resized = tf.image.resize(x_test, (224, 224))

# Normalize the pixel values to [0, 1]


In [4]:
x_train_resized = x_train_resized / 255.0
x_test_resized = x_test_resized / 255.0

# One-hot encode the labels


In [5]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Load the MobileNet model, excluding the top fully connected layers


In [6]:
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

17225924/17225924 [==============================] - 0s 0us/step


# Freeze the base model layers


In [7]:
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers


In [8]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dense(128, activation='relu')(x)
output = Dense(10, activation='softmax')(x)  # CIFAR-10 has 10 classes

# Create the final model


In [9]:
model = Model(inputs=base_model.input, outputs=output)

# Compile the model


In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model


In [11]:
model.fit(x_train_resized, y_train, epochs=10, validation_data=(x_test_resized, y_test))

Epoch 1/10
1563/1563 [==============================] - 290s 184ms/step - loss: 0.5294 - accuracy: 0.8166 - val_loss: 0.4425 - val_accuracy: 0.8484
Epoch 2/10
1563/1563 [==============================] - 295s 189ms/step - loss: 0.3867 - accuracy: 0.8652 - val_loss: 0.4088 - val_accuracy: 0.8574
Epoch 3/10
1563/1563 [==============================] - 304s 194ms/step - loss: 0.3330 - accuracy: 0.8841 - val_loss: 0.3985 - val_accuracy: 0.8648
Epoch 4/10
1563/1563 [==============================] - 307s 197ms/step - loss: 0.2927 - accuracy: 0.8969 - val_loss: 0.4102 - val_accuracy: 0.8629
Epoch 5/10
1563/1563 [==============================] - 315s 202ms/step - loss: 0.2552 - accuracy: 0.9092 - val_loss: 0.4082 - val_accuracy: 0.8682
Epoch 6/10
1563/1563 [==============================] - 314s 201ms/step - loss: 0.2216 - accuracy: 0.9206 - val_loss: 0.4293 - val_accuracy: 0.8676
Epoch 7/10
1563/1563 [==============================] - 311s 199ms/step - loss: 0.1904 - accuracy: 0.9316 - val_

# Evaluate the model


In [12]:
loss, accuracy = model.evaluate(x_test_resized, y_test)
print(f"MobileNet - Loss: {loss}, Accuracy: {accuracy}")

313/313 [==============================] - 49s 155ms/step - loss: 0.5368 - accuracy: 0.8631
MobileNet - Loss: 0.536766767501831, Accuracy: 0.863099992275238
